# GANForge -- StackGAN (Standalone Notebook)

**Text-to-image generation (two-stage)**

This is a **self-contained extract** of the StackGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="7"></a>
## 7. StackGAN -- Text-to-image generation (two-stage)

### 7.1 Problem Statement & Why this GAN

Zhang et al. (2017) tackled **text-to-image synthesis**: given a natural
-language description, generate a matching photo -- a problem plain
class-conditional GANs cannot address at all, since a sentence is a far
richer, more variable condition than a fixed class id. StackGAN's key
ideas are (1) **two stages**: Stage-I sketches low-resolution shape/color
from text, Stage-II takes that sketch *and* the text again and refines it
to a higher-resolution, more detailed image; and (2) **Conditioning
Augmentation (CA)**: instead of feeding the raw text embedding directly,
sample a condition vector `c ~ N(mu(t), sigma(t))` from a small Gaussian
predicted from the text embedding `t`, regularized toward `N(0, I)` with a
KL term. This smooths the conditioning manifold, so nearby (but distinct)
sentences produce smoothly varying images instead of a sparse, easily
-overfit set of exact-caption reproductions.

We build genuine template-generated captions ("a photo of a airplane",
"a small dog", ...) for CIFAR-10 images and a real trainable
Embedding+LSTM text encoder, since paired caption datasets (CUB-200,
Oxford-102) are large downloads inappropriate for a guaranteed-offline
demo. **Swap `stackgan_dataset` for CUB-200-2011 (with its 10
human-written captions per image) for a real text-to-image result.**

### 7.2 Data Collection & Preparation

In [ ]:
STACKGAN_CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
STACKGAN_TEMPLATES = ["a photo of a {}", "a small {}", "an image showing a {}", "a blurry photo of a {}"]
STAGE1_SIZE, STAGE2_SIZE = 32, 64
STACKGAN_MAX_LEN = 8
STACKGAN_NZ, STACKGAN_C_DIM = 100, 64
STACKGAN_BATCH_SIZE = 64
STACKGAN_STAGE1_EPOCHS, STACKGAN_STAGE2_EPOCHS = 6, 6

# ---- Build a tiny vocabulary from the templates themselves ----
_all_words = set()
for t in STACKGAN_TEMPLATES:
    _all_words.update(t.replace("{}", "").split())
_all_words.update(STACKGAN_CLASS_NAMES)
STACKGAN_VOCAB = {"<pad>": 0, "<unk>": 1}
for w in sorted(_all_words):
    STACKGAN_VOCAB[w] = len(STACKGAN_VOCAB)

def tokenize_caption(caption: str) -> torch.Tensor:
    ids = [STACKGAN_VOCAB.get(w, STACKGAN_VOCAB["<unk>"]) for w in caption.split()]
    ids = ids[:STACKGAN_MAX_LEN] + [0] * max(0, STACKGAN_MAX_LEN - len(ids))
    return torch.tensor(ids, dtype=torch.long)

try:
    stackgan_base = torchvision.datasets.CIFAR10(
        root="data", train=True, download=True,
        transform=T.Compose([T.Resize(STAGE2_SIZE), T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)]))
    print(f"Loaded real CIFAR-10 (as image side of synthetic captions): {len(stackgan_base)} images")
except Exception as e:
    print(f"CIFAR-10 download failed ({e}); falling back to synthetic per-class textures.")
    rng = np.random.default_rng(SEED)
    imgs, labels = [], []
    for c in range(10):
        color = torch.tensor(rng.uniform(-1, 1, 3), dtype=torch.float32)   # dtype pinned -- see BigGAN section 6.2 for why
        for _ in range(200):
            img = (torch.randn(3, STAGE2_SIZE, STAGE2_SIZE) * 0.1 + color.view(3, 1, 1)).clamp(-1, 1)
            imgs.append(img); labels.append(c)
    stackgan_base = TensorDataset(torch.stack(imgs), torch.tensor(labels))

class StackGANTextDataset(Dataset):
    '''Wraps an image+label dataset with a randomly-templated CAPTION per
    __getitem__ call, tokenized to fixed length -- the "text" half of the
    text-to-image pair. Real StackGAN training data ships several
    human-written captions per image instead of a template.'''
    def __init__(self, base, class_names, rng_seed=SEED):
        self.base = base
        self.class_names = class_names
        self.rng = np.random.default_rng(rng_seed)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        img, y = self.base[i]
        template = STACKGAN_TEMPLATES[self.rng.integers(len(STACKGAN_TEMPLATES))]
        caption = template.format(self.class_names[int(y)])
        return img, tokenize_caption(caption), int(y)

stackgan_dataset = StackGANTextDataset(stackgan_base, STACKGAN_CLASS_NAMES)
stackgan_loader = DataLoader(stackgan_dataset, batch_size=STACKGAN_BATCH_SIZE, shuffle=True, drop_last=True)

imgs64, tokens, ys = next(iter(stackgan_loader))
sample_captions = [" ".join(k for k, v in STACKGAN_VOCAB.items() if v in tok.tolist() and v > 1) for tok in tokens[:4]]
print("Example (image, caption) pairs:", list(zip([STACKGAN_CLASS_NAMES[y] for y in ys[:4].tolist()], sample_captions)))
show_images(imgs64[:16], title="StackGAN training images (64x64 target)", nrow=4)


### 7.3 Model Architecture

In [ ]:
class StackGAN_TextEncoder(nn.Module):
    '''A small trainable Embedding + LSTM that reads the tokenized caption
    and produces a single fixed-size sentence embedding (its final hidden
    state). The real StackGAN paper uses a pretrained char-CNN-RNN text
    encoder trained jointly with a matching image encoder (Reed et al.);
    we train ours end-to-end with the GANs instead, which is simpler but
    otherwise plays the identical architectural role.'''
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

    def forward(self, tokens):
        emb = self.embed(tokens)
        _, (h_n, _) = self.lstm(emb)
        return h_n[-1]   # (B, hidden_dim)


class ConditioningAugmentation(nn.Module):
    '''Reparameterization-trick sampling: c = mu(t) + sigma(t) * eps. The
    KL(N(mu,sigma) || N(0,I)) term returned alongside `c` is added to the
    generator loss -- it keeps the learned (mu, sigma) close to a standard
    Gaussian, which is what makes the conditioning manifold smooth and
    prevents the network from encoding each caption as an isolated point.'''
    def __init__(self, text_dim=128, c_dim=STACKGAN_C_DIM):
        super().__init__()
        self.fc = nn.Linear(text_dim, c_dim * 2)

    def forward(self, text_embed):
        mu, logvar = self.fc(text_embed).chunk(2, dim=1)
        std = torch.exp(0.5 * logvar)
        c = mu + std * torch.randn_like(std)
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return c, kl


class StackGAN_ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch), nn.ReLU(True),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.BatchNorm2d(ch),
        )

    def forward(self, x):
        return F.relu(x + self.block(x))


class StackGAN_Stage1_Generator(nn.Module):
    '''z (randomness) + c (text-derived condition) -> a coarse 32x32 image
    that captures rough shape and color implied by the caption.'''
    def __init__(self, nz=STACKGAN_NZ, c_dim=STACKGAN_C_DIM, ngf=128):
        super().__init__()
        self.ngf = ngf
        self.fc = nn.Sequential(nn.Linear(nz + c_dim, ngf * 4 * 4 * 4), nn.BatchNorm1d(ngf * 4 * 4 * 4), nn.ReLU(True))
        self.net = nn.Sequential(
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),  # 4 -> 8
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1), nn.BatchNorm2d(ngf), nn.ReLU(True),          # 8 -> 16
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1), nn.BatchNorm2d(ngf // 2), nn.ReLU(True),    # 16 -> 32
            nn.Conv2d(ngf // 2, 3, 3, 1, 1), nn.Tanh(),
        )

    def forward(self, z, c):
        x = self.fc(torch.cat([z, c], dim=1)).view(-1, self.ngf * 4, 4, 4)
        return self.net(x)


class StackGAN_Stage1_Discriminator(nn.Module):
    def __init__(self, c_dim=STACKGAN_C_DIM, ndf=64):
        super().__init__()
        self.img_encoder = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
        )
        self.joint = nn.Sequential(
            nn.Conv2d(ndf * 4 + c_dim, ndf * 4, 1), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 4, 1, 4),
        )

    def forward(self, img, c):
        feat = self.img_encoder(img)
        c_map = c.view(c.size(0), -1, 1, 1).expand(-1, -1, feat.size(2), feat.size(3))
        return self.joint(torch.cat([feat, c_map], dim=1)).view(-1)


class StackGAN_Stage2_Generator(nn.Module):
    '''Takes the STAGE-I image (not noise!) plus the text condition c again,
    encodes it down, fuses in c a second time, refines through residual
    blocks, then upsamples to 64x64 -- adding detail/resolution while
    staying faithful to Stage-I's coarse layout and the original caption.'''
    def __init__(self, c_dim=STACKGAN_C_DIM, ngf=128, n_res_blocks=2):
        super().__init__()
        self.down = nn.Sequential(
            nn.Conv2d(3, ngf, 3, 1, 1), nn.ReLU(True),
            nn.Conv2d(ngf, ngf * 2, 4, 2, 1), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),   # 32 -> 16
        )
        self.fuse = nn.Sequential(nn.Conv2d(ngf * 2 + c_dim, ngf * 2, 3, 1, 1), nn.BatchNorm2d(ngf * 2), nn.ReLU(True))
        self.res_blocks = nn.Sequential(*[StackGAN_ResBlock(ngf * 2) for _ in range(n_res_blocks)])
        self.up = nn.Sequential(
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1), nn.BatchNorm2d(ngf), nn.ReLU(True),        # 16 -> 32
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1), nn.BatchNorm2d(ngf // 2), nn.ReLU(True),  # 32 -> 64
            nn.Conv2d(ngf // 2, 3, 3, 1, 1), nn.Tanh(),
        )

    def forward(self, stage1_img, c):
        feat = self.down(stage1_img)
        c_map = c.view(c.size(0), -1, 1, 1).expand(-1, -1, feat.size(2), feat.size(3))
        feat = self.fuse(torch.cat([feat, c_map], dim=1))
        feat = self.res_blocks(feat)
        return self.up(feat)


class StackGAN_Stage2_Discriminator(nn.Module):
    def __init__(self, c_dim=STACKGAN_C_DIM, ndf=64):
        super().__init__()
        self.img_encoder = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1), nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, True),
        )
        self.joint = nn.Sequential(
            nn.Conv2d(ndf * 8 + c_dim, ndf * 8, 1), nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 8, 1, 4),
        )

    def forward(self, img, c):
        feat = self.img_encoder(img)
        c_map = c.view(c.size(0), -1, 1, 1).expand(-1, -1, feat.size(2), feat.size(3))
        return self.joint(torch.cat([feat, c_map], dim=1)).view(-1)


stackgan_text_encoder = StackGAN_TextEncoder(len(STACKGAN_VOCAB)).to(DEVICE)
stackgan_ca = ConditioningAugmentation().to(DEVICE)
G1_stackgan = StackGAN_Stage1_Generator().to(DEVICE).apply(weights_init)
D1_stackgan = StackGAN_Stage1_Discriminator().to(DEVICE).apply(weights_init)
G2_stackgan = StackGAN_Stage2_Generator().to(DEVICE).apply(weights_init)
D2_stackgan = StackGAN_Stage2_Discriminator().to(DEVICE).apply(weights_init)


### 7.4 Loss Functions & Optimizers

In [ ]:
stackgan_adv_criterion = nn.BCEWithLogitsLoss()
STACKGAN_LAMBDA_KL = 2.0   # weight of the CA KL term relative to the adversarial loss

# Stage-I optimizes G1+text_encoder+CA jointly against D1.
stackgan_opt_G1 = torch.optim.Adam(
    list(G1_stackgan.parameters()) + list(stackgan_text_encoder.parameters()) + list(stackgan_ca.parameters()),
    lr=2e-4, betas=(0.5, 0.999))
stackgan_opt_D1 = torch.optim.Adam(D1_stackgan.parameters(), lr=2e-4, betas=(0.5, 0.999))

# Stage-II optimizes only G2 (Stage-I is frozen; text pipeline reused frozen too -- see 7.5).
stackgan_opt_G2 = torch.optim.Adam(G2_stackgan.parameters(), lr=2e-4, betas=(0.5, 0.999))
stackgan_opt_D2 = torch.optim.Adam(D2_stackgan.parameters(), lr=2e-4, betas=(0.5, 0.999))


### 7.5 Training Loop

In [ ]:
# ---------------------------------------------------------------------------
# 7.5a Stage-I: text -> coarse 32x32 image
# ---------------------------------------------------------------------------
stackgan_stage1_history = []
for epoch in range(STACKGAN_STAGE1_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter(), "KL": AvgMeter()}
    pbar = tqdm(stackgan_loader, desc=f"[StackGAN Stage-I] epoch {epoch+1}/{STACKGAN_STAGE1_EPOCHS}")
    for imgs64, tokens, _ in pbar:
        real64, tokens = imgs64.to(DEVICE), tokens.to(DEVICE)
        real32 = F.interpolate(real64, size=STAGE1_SIZE, mode="bilinear", align_corners=False)
        b = real32.size(0)

        text_embed = stackgan_text_encoder(tokens)
        c, kl = stackgan_ca(text_embed)
        z = torch.randn(b, STACKGAN_NZ, device=DEVICE)
        fake32 = G1_stackgan(z, c)

        # ---- D1 ----
        D1_stackgan.zero_grad(set_to_none=True)
        loss_d1 = stackgan_adv_criterion(D1_stackgan(real32, c.detach()), torch.ones(b, device=DEVICE)) + \
                  stackgan_adv_criterion(D1_stackgan(fake32.detach(), c.detach()), torch.zeros(b, device=DEVICE))
        loss_d1.backward()
        stackgan_opt_D1.step()

        # ---- G1 + text encoder + CA ----
        stackgan_opt_G1.zero_grad(set_to_none=True)
        loss_g1_adv = stackgan_adv_criterion(D1_stackgan(fake32, c), torch.ones(b, device=DEVICE))
        loss_g1 = loss_g1_adv + STACKGAN_LAMBDA_KL * kl
        loss_g1.backward()
        stackgan_opt_G1.step()

        meters["D"].update(loss_d1.item()); meters["G"].update(loss_g1.item()); meters["KL"].update(kl.item())
        pbar.set_postfix(loss_D=f"{loss_d1.item():.3f}", loss_G=f"{loss_g1.item():.3f}", KL=f"{kl.item():.3f}")

    stackgan_stage1_history.append((meters["G"].avg, meters["D"].avg))
    with torch.no_grad():
        G1_stackgan.eval()
        te = stackgan_text_encoder(tokens[:16]); c_demo, _ = stackgan_ca(te)
        sample = G1_stackgan(torch.randn(16, STACKGAN_NZ, device=DEVICE), c_demo)
        G1_stackgan.train()
    show_images(sample, title=f"StackGAN Stage-I samples after epoch {epoch+1}", nrow=4)

save_checkpoint({"G1": G1_stackgan.state_dict(), "D1": D1_stackgan.state_dict(),
                  "text_encoder": stackgan_text_encoder.state_dict(), "ca": stackgan_ca.state_dict()},
                 f"{CHECKPOINT_DIR}/stackgan_stage1.pt")
print("StackGAN Stage-I complete.")


In [ ]:
# ---------------------------------------------------------------------------
# 7.5b Stage-II: (Stage-I image + text) -> refined 64x64 image
# ---------------------------------------------------------------------------
# Stage-I generator and the text pipeline are FROZEN here (requires_grad_(False))
# -- Stage-II's only job is to add detail/resolution conditioned on what
# Stage-I already committed to, exactly as in the original paper.
G1_stackgan.eval()
for p in G1_stackgan.parameters():
    p.requires_grad_(False)
for p in stackgan_text_encoder.parameters():
    p.requires_grad_(False)
for p in stackgan_ca.parameters():
    p.requires_grad_(False)

stackgan_stage2_history = []
for epoch in range(STACKGAN_STAGE2_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter()}
    pbar = tqdm(stackgan_loader, desc=f"[StackGAN Stage-II] epoch {epoch+1}/{STACKGAN_STAGE2_EPOCHS}")
    for imgs64, tokens, _ in pbar:
        real64, tokens = imgs64.to(DEVICE), tokens.to(DEVICE)
        b = real64.size(0)

        with torch.no_grad():
            text_embed = stackgan_text_encoder(tokens)
            c, _ = stackgan_ca(text_embed)
            z = torch.randn(b, STACKGAN_NZ, device=DEVICE)
            stage1_img = G1_stackgan(z, c)
        fake64 = G2_stackgan(stage1_img, c)

        # ---- D2 ----
        D2_stackgan.zero_grad(set_to_none=True)
        loss_d2 = stackgan_adv_criterion(D2_stackgan(real64, c), torch.ones(b, device=DEVICE)) + \
                  stackgan_adv_criterion(D2_stackgan(fake64.detach(), c), torch.zeros(b, device=DEVICE))
        loss_d2.backward()
        stackgan_opt_D2.step()

        # ---- G2 ----
        stackgan_opt_G2.zero_grad(set_to_none=True)
        loss_g2 = stackgan_adv_criterion(D2_stackgan(fake64, c), torch.ones(b, device=DEVICE))
        loss_g2.backward()
        stackgan_opt_G2.step()

        meters["D"].update(loss_d2.item()); meters["G"].update(loss_g2.item())
        pbar.set_postfix(loss_D=f"{loss_d2.item():.3f}", loss_G=f"{loss_g2.item():.3f}")

    stackgan_stage2_history.append((meters["G"].avg, meters["D"].avg))
    with torch.no_grad():
        G2_stackgan.eval()
        sample = G2_stackgan(stage1_img[:16], c[:16])
        G2_stackgan.train()
    show_images(sample, title=f"StackGAN Stage-II samples after epoch {epoch+1}", nrow=4)

save_checkpoint({"G2": G2_stackgan.state_dict(), "D2": D2_stackgan.state_dict()}, f"{CHECKPOINT_DIR}/stackgan_stage2.pt")
print("StackGAN Stage-II complete.")


### 7.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def stackgan_generate(caption: str, n_samples: int = 4, seed: int = None):
    '''End-to-end text -> (stage1_image, stage2_image). This is the function
    a real product would call: user types a sentence, gets an image back.'''
    if seed is not None:
        torch.manual_seed(seed)
    tokens = tokenize_caption(caption).unsqueeze(0).repeat(n_samples, 1).to(DEVICE)
    G1_stackgan.eval(); G2_stackgan.eval()
    text_embed = stackgan_text_encoder(tokens)
    c, _ = stackgan_ca(text_embed)
    z = torch.randn(n_samples, STACKGAN_NZ, device=DEVICE)
    stage1_img = G1_stackgan(z, c)
    stage2_img = G2_stackgan(stage1_img, c)
    G1_stackgan.train(); G2_stackgan.train()
    return stage1_img, stage2_img

s1, s2 = stackgan_generate("a photo of a horse", n_samples=8)
show_images(F.interpolate(s1, size=STAGE2_SIZE), title="Stage-I (upsampled for display)", nrow=8, figsize=(12, 2))
show_images(s2, title="Stage-II (final)", nrow=8, figsize=(12, 2))


### 7.7 Visualization & Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
g1s, d1s = zip(*stackgan_stage1_history); g2s, d2s = zip(*stackgan_stage2_history)
axes[0].plot(g1s, label="G1"); axes[0].plot(d1s, label="D1"); axes[0].legend(); axes[0].set_title("Stage-I losses")
axes[1].plot(g2s, label="G2"); axes[1].plot(d2s, label="D2"); axes[1].legend(); axes[1].set_title("Stage-II losses")
plt.show()

# Side-by-side: for several captions, show Stage-I vs Stage-II output --
# directly visualizes what the refinement stage adds.
fig, axes = plt.subplots(3, 2, figsize=(5, 8))
for row, cap in enumerate(["a photo of a bird", "a small dog", "an image showing a ship"]):
    s1, s2 = stackgan_generate(cap, n_samples=1, seed=row)
    axes[row, 0].imshow(denorm(F.interpolate(s1, size=STAGE2_SIZE))[0].permute(1, 2, 0).cpu()); axes[row, 0].set_title(f"Stage-I: '{cap}'", fontsize=8); axes[row, 0].axis("off")
    axes[row, 1].imshow(denorm(s2)[0].permute(1, 2, 0).cpu()); axes[row, 1].set_title("Stage-II", fontsize=8); axes[row, 1].axis("off")
plt.tight_layout(); plt.show()

# Conditioning-augmentation smoothness check: same caption, resampled c
# each time -- outputs should vary smoothly, not jump between unrelated
# images, if the CA/KL regularization is doing its job.
s1_multi, s2_multi = stackgan_generate("a photo of a cat", n_samples=8)
show_images(s2_multi, title="Same caption, 8 resampled conditioning vectors c", nrow=8, figsize=(12, 2))


### 7.8 User Input → Output Demo

In [ ]:
USER_CAPTION = "a photo of a truck"
N_SAMPLES_TO_GENERATE = 4

_, s2_demo = stackgan_generate(USER_CAPTION, n_samples=N_SAMPLES_TO_GENERATE, seed=42)
show_images(s2_demo, title=f"User caption: '{USER_CAPTION}'", nrow=4)


### 7.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Conditioning Augmentation is a general trick, not
  StackGAN-specific -- any GAN conditioned on a continuous, high
  -dimensional, sparsely-sampled embedding (text, audio, even
  fine-grained attribute vectors) benefits from smoothing that embedding
  with a small VAE-style reparameterized sample before it reaches the
  generator.
- ⚠️ **Warning.** With only template-generated captions ("a photo of a
  {class}"), the model can only ever really recover *class identity* from
  text -- true free-form text-to-image (arbitrary attributes, colors,
  counts, relations) requires a real captioned dataset (CUB-200,
  Oxford-102 Flowers, or COCO) with genuine linguistic variation.
- ⚠️ **Warning.** Because Stage-II is trained conditioned on Stage-I's
  *own* outputs (not ground truth), any systematic bias Stage-I develops
  (e.g. always centering the subject) propagates into Stage-II -- this
  coupling is why later text-to-image work (AttnGAN, diffusion models)
  moved toward single-stage, attention-based refinement instead of a
  two-stage pipeline.

**This GAN solved the problem of: text-to-image generation -- turning a
free-form natural-language description into a matching image, first
sketched coarsely then refined, rather than picking from a fixed set of
classes.**